In [ ]:
import pandas as pd
from pathlib import Path
DATA_PATH = (Path.cwd().parent.parent.parent / "train_ids.csv")
df = pd.read_csv(DATA_PATH)

# Metadata EDA 
df.info()
df.isna().sum().sort_values(ascending=False)

In [ ]:
pathology_cols = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly",
    "Lung Opacity", "Pneumonia", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices",
]

label_counts = {}
for col in pathology_cols:
    counts = df[col].value_counts(dropna=False).sort_index()
    label_counts[col] = {
        "positive (1)": counts.get(1.0, 0),
        "uncertain (0)": counts.get(0.0, 0),
        "negative (-1)": counts.get(-1.0, 0),
        "NaN": df[col].isna().sum(),
    }

label_df = pd.DataFrame(label_counts).T
label_df["total_labeled"] = label_df[["positive (1)", "uncertain (0)", "negative (-1)"]].sum(axis=1)
label_df

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

plot_df = label_df[["positive (1)", "uncertain (0)", "negative (-1)", "NaN"]]
plot_df.plot(kind="barh", stacked=True, ax=ax,
             color=["#2ecc71", "#f39c12", "#e74c3c", "#bdc3c7"])

ax.set_xlabel("Number of Images")
ax.set_title("Label Distribution per Pathology")
ax.legend(title="Label", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
print(df["Frontal/Lateral"].value_counts(dropna=False))
print(df["AP/PA"].value_counts(dropna=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["Age"], bins=30, color="#3498db", edgecolor="white", alpha=0.85)
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Number of Images")
axes[0].set_title("Age Distribution")
axes[0].axvline(df["Age"].median(), color="#e74c3c", linestyle="--", label=f'Median: {df["Age"].median():.0f}')
axes[0].legend()

sex_counts = df["Sex"].value_counts()
bars = axes[1].bar(sex_counts.index, sex_counts.values, color=["#3498db", "#e74c3c"], edgecolor="white")
axes[1].set_xlabel("Sex")
axes[1].set_ylabel("Number of Images")
axes[1].set_title("Sex Distribution")
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1000,
                 f'{bar.get_height():,}', ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()


In [ ]:
df["patient_id"] = df["Path"].str.split("/").str[1]
print(f"Total Images: {len(df)}")
print(f"Unique Patients: {df['patient_id'].nunique()}")
df["patient_id"].value_counts().describe()
img_per_patient = df["patient_id"].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(img_per_patient, bins=50, color="#3498db", edgecolor="white")
ax.set_xlabel("Images per Patient")
ax.set_ylabel("Number of Patients")
ax.set_title("Distribution of Images per Patient")
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

corr = df[pathology_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Pathology Correlation Matrix")
plt.tight_layout()
plt.show()